In [1]:
import pytesseract
import cv2

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [28]:
# Get the dimensions of the text
text_width, text_height = cv2.getTextSize("anonymized using resumefire.io", cv2.FONT_HERSHEY_SIMPLEX, 1, 1)[0]

image = cv2.imread("./resume.jpeg")
height, width, _ = image.shape

# Calculate the bottom-right corner coordinates
x = width - text_width - 50  # Adjust the 10 for padding
y = height - text_height - 50  # Adjust the 10 for padding

cv2.putText(image, "anonymized using resumefire.io", (x, y), cv2.FONT_HERSHEY_DUPLEX, 1, (0, 0, 0), 1, cv2.LINE_AA)
image = cv2.resize(image, (637, 825))
cv2.imshow('thresh', image)
# cv2.imshow('opening', opening)
# cv2.imshow('invert', invert)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [18]:
image = cv2.imread("./resume-0.jpeg")

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (3,3), 0)
thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_OTSU)[1]

d = pytesseract.image_to_data(thresh, output_type=pytesseract.Output.DATAFRAME, config='--psm 6')

In [27]:
allowed_single_letters = set(['i', 'a', '@', '&'])
bounding_boxes = []
n_boxes = d.shape[0]
for i in range(n_boxes):
    (x, y, w, h) = (d["left"][i], d["top"][i], d["width"][i], d["height"][i])
    text = str(d["text"][i]).strip()
    if d["level"][i]==5 and (len(text)>1 or text.lower() in allowed_single_letters) and d["conf"][i] >= 60:
        bounding_boxes.append((int(x), int(y), int(w), int(h)))
        # cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)

# image = cv2.resize(image, (637, 825))
cv2.imshow('thresh', image)
# cv2.imshow('opening', opening)
# cv2.imshow('invert', invert)
cv2.waitKey(0)
cv2.destroyAllWindows()
height, width = image.shape[:2]
height, width
bounding_boxes


[{'x': 516, 'y': 100, 'w': 172, 'h': 49},
 {'x': 717, 'y': 100, 'w': 258, 'h': 62},
 {'x': 1005, 'y': 101, 'w': 180, 'h': 48},
 {'x': 506, 'y': 168, 'w': 155, 'h': 20},
 {'x': 689, 'y': 167, 'w': 261, 'h': 26},
 {'x': 978, 'y': 166, 'w': 217, 'h': 28},
 {'x': 102, 'y': 241, 'w': 171, 'h': 24},
 {'x': 131, 'y': 293, 'w': 156, 'h': 27},
 {'x': 300, 'y': 295, 'w': 28, 'h': 20},
 {'x': 343, 'y': 292, 'w': 107, 'h': 23},
 {'x': 1428, 'y': 292, 'w': 99, 'h': 27},
 {'x': 1541, 'y': 293, 'w': 44, 'h': 21},
 {'x': 132, 'y': 332, 'w': 106, 'h': 20},
 {'x': 249, 'y': 332, 'w': 24, 'h': 26},
 {'x': 281, 'y': 332, 'w': 87, 'h': 21},
 {'x': 380, 'y': 334, 'w': 22, 'h': 18},
 {'x': 415, 'y': 332, 'w': 118, 'h': 26},
 {'x': 542, 'y': 332, 'w': 88, 'h': 21},
 {'x': 1305, 'y': 332, 'w': 52, 'h': 26},
 {'x': 1375, 'y': 333, 'w': 55, 'h': 20},
 {'x': 1461, 'y': 333, 'w': 49, 'h': 19},
 {'x': 1527, 'y': 333, 'w': 56, 'h': 20},
 {'x': 131, 'y': 370, 'w': 74, 'h': 21},
 {'x': 217, 'y': 371, 'w': 61, 'h': 20}

In [28]:
import json
with open('./bounding_boxes.json', 'w') as f:
    obj = {
        "dimensions": {
            "height": height,
            "width": width
        },
        "boxes": bounding_boxes
    }
    f.write(json.dumps(obj))

In [4]:
d.loc[(d['level'] == 5) & (d['conf'] > 85) & (d['width'] > 3)]

,level,page_num,block_num,par_num,line_num,word_num,left,top,width,height,conf,text
4,5,1,1,1,1,1,516,100,172,49,92.002594,Imon
5,5,1,1,1,1,2,717,100,258,62,90.385727,Kallyan
6,5,1,1,1,1,3,1005,101,180,48,96.975601,Tatar
8,5,1,1,1,2,1,506,168,155,20,95.868523,518-704-1397
10,5,1,1,1,2,3,689,167,261,26,92.280716,imontatar@gmail.com
...,...,...,...,...,...,...,...,...,...,...,...,...
401,5,1,1,1,42,12,944,1851,116,26,91.669746,paginated
402,5,1,1,1,42,13,1071,1852,74,19,96.527756,article
403,5,1,1,1,42,14,1156,1851,53,21,96.930542,data
406,5,1,1,1,43,2,197,1890,100,21,92.621368,Website:


In [ ]:
len(d['text'])

In [13]:
n_boxes = len(d["level"])
for i in range(n_boxes):
    (x, y, w, h) = (d["left"][i], d["top"][i], d["width"][i], d["height"][i])
    if d["level"][i]==5 and d["conf"][i] >= 60 and str(d["text"][i]).strip() != "":
        cv2.rectangle(thresh, (x, y), (x + w, y + h), (0, 255, 0), 2)

cv2.imshow('thresh', thresh)
# cv2.imshow('opening', opening)
# cv2.imshow('invert', invert)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [12]:
n_boxes = len(d["level"])
for i in range(n_boxes):
    (x, y, w, h) = (d["left"][i], d["top"][i], d["width"][i], d["height"][i])
    if d["level"][i]==4 and d["conf"][i] == -1:
        cv2.rectangle(thresh, (x, y), (x + w, y + h), (0, 255, 0), 2)

cv2.imshow('thresh', thresh)
# cv2.imshow('opening', opening)
# cv2.imshow('invert', invert)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [9]:
#TODO: stuff to blur out
# 1. name
# 2. email, phone no
# 3 location

In [28]:
cv2.namedWindow("custom window", cv2.WINDOW_NORMAL)
cv2.imshow("custom window", image)
cv2.resizeWindow("custom window", thresh.shape[:2][1], thresh.shape[:2][0])
cv2.waitKey(0)
cv2.destroyAllWindows()

In [29]:
cv2.imshow("resume", image)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [12]:
d.keys()

Index(['level', 'page_num', 'block_num', 'par_num', 'line_num', 'word_num',
       'left', 'top', 'width', 'height', 'conf', 'text'],
      dtype='object')

In [32]:
for i in range(len(d["level"])):
    if d["level"][i] == 5:
        print(d["text"][i], d["line_num"][i], d["block_num"][i], d["par_num"][i], d['left'][i], d['left'][i]+d['width'][i])

Imon 1 1 1 516 688
Kallyan 1 1 1 717 975
Tatar 1 1 1 1005 1185
518-704-1397 2 1 1 506 661
| 2 1 1 673 675
imontatar@gmail.com 2 1 1 689 950
| 2 1 1 963 965
github.com/Ta7ar 2 1 1 978 1195
EDUCATION 3 1 1 102 273
University 4 1 1 131 287
at 4 1 1 300 328
Buffalo 4 1 1 343 450
Buffalo, 4 1 1 1428 1527
NY 4 1 1 1541 1585
Bachelor 5 1 1 132 238
of 5 1 1 249 273
Science 5 1 1 281 368
in 5 1 1 380 402
Computer 5 1 1 415 533
Science 5 1 1 542 630
Aug. 5 1 1 1305 1357
2019 5 1 1 1375 1430
- 5 1 1 1441 1448
Dec. 5 1 1 1461 1510
2022 5 1 1 1527 1583
GPA: 6 1 1 131 205
3.892 6 1 1 217 278
Coursework: 7 1 1 131 302
Datastructures 7 1 1 317 495
& 7 1 1 506 525
Algorithms, 7 1 1 537 676
Intro. 7 1 1 689 753
to 7 1 1 768 792
Machine 7 1 1 803 902
Learning, 7 1 1 913 1024
Intro. 7 1 1 1037 1101
to 7 1 1 1116 1140
Web 7 1 1 1150 1203
Applications, 7 1 1 1214 1370
Probability 7 1 1 1382 1516
& 7 1 1 1527 1546
Statistical 8 1 1 131 251
Inference, 8 1 1 262 376
Linear 8 1 1 388 464
Algebra, 8 1 1 475 574
